# Step 09b — Compute Tukey IQR-based LOC cap

> Compute the LOC cap used by Step 10 to bound ground_truth sampling. Applies Tukey's IQR upper fence on the `file_line_count` column of `data/9_ground_truth_candidates.csv`. The cap excludes structurally atypical (very large) workflows and bounds LLM token cost in downstream agent evaluation.

**Paper section:** Step 3 — Ground-truth construction / Stratified sampling (LOC cap derivation).

## Method
Compute on `file_line_count` over all candidate rows:

```
Q1          = 25th percentile
Q3          = 75th percentile
IQR         = Q3 - Q1
upper_fence = Q3 + 1.5 * IQR
lower_fence = Q1 - 1.5 * IQR   # descriptive only; LOC minimum is undefined per study design
cap         = ceil(upper_fence)
```

Step 10 drops rows with `file_line_count > cap` before stratified quartile sampling.

## Inputs
| Path | Format | Source |
|---|---|---|
| `data/9_ground_truth_candidates.csv` | CSV (one row per workflow file) | Step 09 |
| `data/step9_report.json` | JSON | Step 09 (integrity chain) |

## Outputs
| Path | Format | Consumer |
|---|---|---|
| `data/9b_loc_cap.json` | JSON (cap value + IQR statistics) | Step 10 |
| `data/step9b_report.json` | JSON | Step 10 (integrity chain) |
| `data/figures/loc_distribution.png` | PNG (histogram + boxplot) | Paper / inspection |


In [1]:
# --- Notebook bootstrap ---
# Idempotent: chdir to repo root + reset sys.argv for argparse.
from dataset_utils.notebook_bootstrap import setup

setup()

print("Step 09b — starting: compute Tukey IQR-based LOC cap...", flush=True)
"""Step 09b: Compute Tukey IQR LOC cap for ground_truth sampling.

Reads data/9_ground_truth_candidates.csv (output of Step 09), computes the
Tukey IQR upper fence on the `file_line_count` column, and emits
data/9b_loc_cap.json — consumed by Step 10 to bound sampled workflow size.

Inputs:
  data/9_ground_truth_candidates.csv     Step 09 output
  data/step9_report.json           Step 09 report (integrity chain)

Outputs:
  data/9b_loc_cap.json                 Cap value + IQR statistics
  data/step9b_report.json           Audit report
"""

import argparse
import json
import math
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl


def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(
        description="Compute Tukey IQR-based LOC cap for ground_truth sampling."
    )
    p.add_argument(
        "--input",
        type=Path,
        default=Path("data/9_ground_truth_candidates.csv"),
        help="Ground truth candidates CSV from Step 09 (default: data/9_ground_truth_candidates.csv).",
    )
    p.add_argument(
        "--output-cap",
        type=Path,
        default=Path("data/9b_loc_cap.json"),
        help="Output JSON with cap value + statistics (default: data/9b_loc_cap.json).",
    )
    p.add_argument(
        "--report",
        type=Path,
        default=Path("data/step9b_report.json"),
        help="Output audit report (default: data/step9b_report.json).",
    )
    p.add_argument(
        "--figure",
        type=Path,
        default=Path("data/figures/loc_distribution.png"),
        help="Output PNG figure (default: data/figures/loc_distribution.png).",
    )
    p.add_argument(
        "--prior-report",
        type=Path,
        default=Path("data/step9_report.json"),
        help="Step 09 report JSON for integrity chain (default: data/step9_report.json).",
    )
    p.add_argument(
        "--no-display",
        action="store_true",
        help="Skip plt.show() — useful for scripted execution.",
    )
    return p.parse_args()


def compute_cap(loc_values: np.ndarray) -> dict:
    """Tukey IQR fence + integer cap.

    Returns: q1, median, q3, iqr, upper_fence, lower_fence, cap.
    cap = ceil(upper_fence). lower_fence is descriptive only — LOC
    minimum is undefined per study design.
    """
    q1 = float(np.percentile(loc_values, 25))
    median = float(np.percentile(loc_values, 50))
    q3 = float(np.percentile(loc_values, 75))
    iqr = q3 - q1
    upper_fence = q3 + 1.5 * iqr
    lower_fence = q1 - 1.5 * iqr
    cap = int(math.ceil(upper_fence))
    return {
        "q1": q1,
        "median": median,
        "q3": q3,
        "iqr": iqr,
        "upper_fence": upper_fence,
        "lower_fence": lower_fence,
        "cap": cap,
    }


def percentile_table(loc_values: np.ndarray) -> dict[str, float]:
    pct_points = [10, 25, 50, 75, 90, 95, 99]
    return {f"p{p}": float(np.percentile(loc_values, p)) for p in pct_points}


def print_console_summary(
    *,
    n_total: int,
    n_zero: int,
    cap_stats: dict,
    pct: dict[str, float],
    excluded_count: int,
    retained_count: int,
    top10: list[tuple[str, int]],
) -> None:
    cap = cap_stats["cap"]
    excl_pct = excluded_count / n_total * 100
    ret_pct = retained_count / n_total * 100

    print()
    print("=" * 72)
    print("  Step 09b — Tukey IQR LOC cap")
    print("=" * 72)
    print()
    print(f"  Input rows           : {n_total:,}")
    print(f"  Zero-LOC rows        : {n_zero:,}")
    print()
    print("  LOC percentiles:")
    for k, v in pct.items():
        print(f"    {k:>4} : {v:>10,.1f}")
    print()
    print(f"LOC Max. (Cap) =  {cap}")
    print("Workflows:")
    print(f"excluded (LOC > {cap}):    {excluded_count:,}  ({excl_pct:5.2f}%)")
    print(f"retained (LOC <= {cap}):   {retained_count:,}  ({ret_pct:5.2f}%)")
    print("  Tukey IQR fence:")
    print(f"    Q1          : {cap_stats['q1']:>10,.2f}")
    print(f"    Median      : {cap_stats['median']:>10,.2f}")
    print(f"    Q3          : {cap_stats['q3']:>10,.2f}")
    print(f"    IQR         : {cap_stats['iqr']:>10,.2f}")
    print(f"    upper_fence : {cap_stats['upper_fence']:>10,.2f}")
    print()
    print()
    print("  Top 10 largest workflows (by LOC):")
    print(f"    {'LOC':>8}  file_key")
    print(f"    {'-' * 8}  {'-' * 60}")
    for fk, loc in top10:
        print(f"    {loc:>8,}  {fk[:70]}")
    print()


def make_figure(loc_values: np.ndarray, cap: int, output: Path, *, display: bool = True) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # X-axis upper bound: tail-clipped for legibility.
    upper = max(int(np.percentile(loc_values, 99)) * 2, cap * 2)

    ax1 = axes[0]
    bins = np.linspace(0, upper, 60)
    ax1.hist(np.clip(loc_values, 0, upper), bins=bins, color="#4477aa", edgecolor="white")
    ax1.axvline(cap, color="#cc3311", linestyle="--", linewidth=2, label=f"cap = {cap}")
    ax1.set_xlabel("LOC (file_line_count, clipped at p99×2)")
    ax1.set_ylabel("count (log scale)")
    ax1.set_yscale("log")
    ax1.set_title("LOC distribution")
    ax1.legend()

    ax2 = axes[1]
    ax2.boxplot(
        loc_values,
        vert=False,
        showfliers=True,
        flierprops=dict(
            marker="o", markersize=3, markerfacecolor="#999999",
            markeredgecolor="#999999", linestyle="none", alpha=0.4,
        ),
    )
    ax2.axvline(cap, color="#cc3311", linestyle="--", linewidth=2, label=f"cap = {cap}")
    ax2.set_xlim(0, upper)
    ax2.set_xlabel("LOC")
    ax2.set_yticks([])
    ax2.set_title("LOC boxplot (Tukey IQR fences = whiskers)")
    ax2.legend()

    fig.suptitle("Step 09b — LOC distribution and Tukey IQR cap", fontsize=14)
    fig.tight_layout()

    output.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output, dpi=150, bbox_inches="tight")
    if display:
        plt.show()
    plt.close(fig)


def main() -> None:
    args = parse_args()

    try:
        sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
    except NameError:
        pass  # Jupyter: __file__ undefined; dataset_utils on path via wheel install
    from dataset_utils.hashing import compute_file_hash
    from dataset_utils.integrity import assert_input_matches_report

    input_sha = assert_input_matches_report(
        input_path=args.input,
        prior_report_path=args.prior_report,
        label="step 09b input",
    )
    print(f"  input SHA: {input_sha[:16]}...", flush=True)

    print("Reading ground_truth candidates CSV...", flush=True)
    df = pl.read_csv(args.input, encoding="utf-8-sig")
    if df["file_line_count"].dtype == pl.Utf8:
        df = df.with_columns(
            pl.col("file_line_count").cast(pl.Int64, strict=False).fill_null(0)
        )

    loc_series = df["file_line_count"]
    loc_values = loc_series.to_numpy()
    n_total = int(loc_series.len())
    n_zero = int((loc_series == 0).sum())

    cap_stats = compute_cap(loc_values)
    cap = cap_stats["cap"]
    pct = percentile_table(loc_values)

    excluded_count = int((loc_series > cap).sum())
    retained_count = n_total - excluded_count

    top10_iter = (
        df.sort("file_line_count", descending=True)
        .head(10)
        .select(["file_key", "file_line_count"])
        .iter_rows()
    )
    top10_list = [(fk, int(loc)) for fk, loc in top10_iter]

    print_console_summary(
        n_total=n_total,
        n_zero=n_zero,
        cap_stats=cap_stats,
        pct=pct,
        excluded_count=excluded_count,
        retained_count=retained_count,
        top10=top10_list,
    )

    loc_cap_payload = {
        "cap": cap,
        "q1": cap_stats["q1"],
        "median": cap_stats["median"],
        "q3": cap_stats["q3"],
        "iqr": cap_stats["iqr"],
        "upper_fence": cap_stats["upper_fence"],
        "lower_fence": cap_stats["lower_fence"],
        "percentiles": pct,
        "n_total": n_total,
        "n_zero_loc": n_zero,
        "excluded_count": excluded_count,
        "retained_count": retained_count,
        "input_file": str(args.input),
        "input_sha256": input_sha,
    }
    args.output_cap.parent.mkdir(parents=True, exist_ok=True)
    with args.output_cap.open("w", encoding="utf-8") as f:
        json.dump(loc_cap_payload, f, indent=2, ensure_ascii=False)
    cap_sha = compute_file_hash(args.output_cap)
    print(f"Wrote {args.output_cap} (sha {cap_sha[:16]}...)", flush=True)

    report = {
        "step": "09b",
        "description": "Tukey IQR LOC cap for ground_truth sampling",
        "input_file": str(args.input),
        "input_sha256": input_sha,
        "prior_report": str(args.prior_report),
        "n_total": n_total,
        "n_zero_loc": n_zero,
        "cap": cap,
        "cap_derivation": {
            "method": "Tukey IQR upper fence",
            "formula": "ceil(Q3 + 1.5 * IQR)",
            "q1": cap_stats["q1"],
            "q3": cap_stats["q3"],
            "iqr": cap_stats["iqr"],
            "upper_fence": cap_stats["upper_fence"],
            "lower_fence": cap_stats["lower_fence"],
        },
        "exclusion_summary": {
            "excluded_count": excluded_count,
            "excluded_pct": round(excluded_count / n_total * 100, 4),
            "retained_count": retained_count,
            "retained_pct": round(retained_count / n_total * 100, 4),
        },
        "percentiles": pct,
        "output_file": str(args.output_cap),
        "output_sha256": cap_sha,
    }
    args.report.parent.mkdir(parents=True, exist_ok=True)
    with args.report.open("w", encoding="utf-8") as f:
        json.dump(report, f, indent=2, ensure_ascii=False)
    print(f"Wrote {args.report}", flush=True)
    print()
    print("Step 09b complete.")


main()


Step 09b — starting: compute Tukey IQR-based LOC cap...


  input SHA: 3ce2f38cc05171af...


Reading ground_truth candidates CSV...



  Step 09b — Tukey IQR LOC cap

  Input rows           : 49,829
  Zero-LOC rows        : 0

  LOC percentiles:
     p10 :       23.0
     p25 :       34.0
     p50 :       58.0
     p75 :      104.0
     p90 :      203.0
     p95 :      303.0
     p99 :      660.4

LOC Max. (Cap) =  209
Workflows:
excluded (LOC > 209):    4,716  ( 9.46%)
retained (LOC <= 209):   45,113  (90.54%)
  Tukey IQR fence:
    Q1          :      34.00
    Median      :      58.00
    Q3          :     104.00
    IQR         :      70.00
    upper_fence :     209.00


  Top 10 largest workflows (by LOC):
         LOC  file_key
    --------  ------------------------------------------------------------
      13,199  opendds@opendds@build_and_test.yml
       8,112  microsoft@openvmm@openvmm-ci.yaml
       8,037  microsoft@openvmm@openvmm-pr.yaml
       7,579  microsoft@openvmm@openvmm-pr-release.yaml
       5,244  siderolabs@talos@ci.yaml
       5,213  appsignal@appsignal-ruby@ci.yml
       5,120  replicatedhq@kot

Wrote data/step9b_report.json



Step 09b complete.


---

**Next step:** [`10_sample_ground_truth.ipynb`](./10_sample_ground_truth.ipynb)
